In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=spark.table('test_exp_bricks.default.address')

In [0]:
display(df)

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://synapse@testexpstorageacc.dfs.core.windows.net/databrickstable/")

In [0]:
df.write.format("delta")\
.option("path","abfss://synapse@testexpstorageacc.dfs.core.windows.net/databrickstable/address/")\
.saveAsTable("test_exp_bricks.default.address_delta")


In [0]:
%sql
CREATE TABLE test_exp_bricks.testdata.employee_raw (
    id int,empname varchar(50),emplocation varchar(50), empsalary bigint,empgender char(1)
)
USING DELTA
LOCATION "abfss://synapse@testexpstorageacc.dfs.core.windows.net/databrickstable/employee_raw/"

In [0]:
%sql
CREATE TABLE test_exp_bricks.testdata.employee_bronze(
    id int,empname varchar(50),emplocation varchar(50), empsalary bigint,empgender char(1),currind int
)
USING DELTA
LOCATION "abfss://synapse@testexpstorageacc.dfs.core.windows.net/databrickstable/employee_bronze/"

In [0]:
%sql
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (101,'manesh','trv',50000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (102,'nimesh','cok',70000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (103,'rajani','ekm',40200,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (104,'raju','lax',25000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (105,'rahul','dxb',11000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (106,'midhu','dxb',63000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (107,'arshiq','cok',84000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (108,'jilu','maa',65000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (109,'jebin','igi',4000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (110,'meril','mum',45400,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (111,'riya','igi',5800,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (112,'liss','cok',7000,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (113,'aysha','mum',6000,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (114,'achu','fra',4600,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (115,'arya','jfk',5900,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (116,'karthi','ont',7000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (117,'anvi','jfk',3000,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (118,'alok','del',15000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (119,'arathi','del',4500,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (120,'arun','blr',5000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (121,'anu','blr',7000,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (122,'zara','blr',12000,'F');

In [0]:
%sql

INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (101,'manesh','kkm',57000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (102,'nimesh','cok',78000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (103,'rajani','epr',44200,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (104,'raju','kka',26600,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (123,'sony','trv',12500,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (124,'philip','blr',21000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (125,'jen','cok',25050,'F');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (126,'michael','ama',68000,'M');
INSERT INTO test_exp_bricks.testdata.employee_raw VALUES (127,'ligin','jfk',19000,'M');


In [0]:
%sql
OPTIMIZE test_exp_bricks.testdata.employee_raw

In [0]:
%sql
DESCRIBE HISTORY test_exp_bricks.testdata.employee_raw

In [0]:
%sql
SELECT * from test_exp_bricks.testdata.employee_raw

In [0]:
def execute_merge_type2 (tgttable, srctable):
    tgtdf=spark.table(tgttable)
    srcdf=spark.table(srctable)

    srcmatchrecords= srcdf.join(tgtdf,srcdf.id==tgtdf.id,"left_semi")
    srcmatchrecords.createOrReplaceTempView("srcmatchrecords")

    if srcmatchrecords.count():
        updatecode = f'MERGE INTO {tgttable} AS tgt USING srcmatchrecords AS src ON src.id=tgt.id and tgt.currind=1 WHEN MATCHED THEN UPDATE SET currind=0'
        spark.sql(updatecode)

    srcdf=srcdf.withColumn("currind",lit(1).cast("int"))

    srcdf.write.mode("append").saveAsTable(tgttable)


In [0]:
execute_merge_type2 ("test_exp_bricks.testdata.employee_bronze","test_exp_bricks.testdata.employee_raw")

In [0]:
%sql
select * from test_exp_bricks.testdata.employee_raw

In [0]:
%sql
select * from test_exp_bricks.testdata.employee_bronze

In [0]:
%sql
MERGE INTO test_exp_bricks.testdata.employee_bronze AS tgt 
USING srcmatchrecords AS src ON src.id=tgt.id and tgt.currind=1 
WHEN MATCHED THEN UPDATE SET currind=0